# Simple Organized Image Manifest

Set `ROOT_FOLDER`, then run the next cell. It creates a neat Excel file with only the useful columns: `data_name`, `path`, and `label`, plus a few basic file details.


In [1]:
# 1) Put your organized folder path here
ROOT_FOLDER = r'/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy'

# 2) Output file name. Leave as None to save inside ROOT_FOLDER.
OUTPUT_XLSX = None


In [2]:
from pathlib import Path
from datetime import datetime
from collections import Counter

try:
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.worksheet.table import Table, TableStyleInfo
    from openpyxl.utils import get_column_letter
except ImportError:
    raise SystemExit("Missing openpyxl. Install it with: conda install openpyxl -y")

root = Path(ROOT_FOLDER).expanduser()
if not root.exists():
    raise FileNotFoundError(f"ROOT_FOLDER does not exist: {root}")

output_path = Path(OUTPUT_XLSX) if OUTPUT_XLSX else root / "organized_image_manifest.xlsx"

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp", ".heic"}
rows = []

for img in sorted(root.rglob("*")):
    if not img.is_file() or img.suffix.lower() not in image_exts:
        continue

    rel = img.relative_to(root)
    label = img.parent.name                      # deepest folder containing the image
    data_name = img.stem                         # filename without extension

    rows.append([
        data_name,
        str(rel),
        label,
        img.name,
        img.suffix.lower(),
        round(img.stat().st_size / 1024, 1),
        datetime.fromtimestamp(img.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
    ])

wb = Workbook()
ws = wb.active
ws.title = "manifest"

headers = ["data_name", "path", "label", "filename", "ext", "size_kb", "modified_time"]
ws.append(headers)
for row in rows:
    ws.append(row)

# Neat formatting
header_fill = PatternFill("solid", fgColor="1F4E78")
for cell in ws[1]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center")

ws.freeze_panes = "A2"
ws.auto_filter.ref = ws.dimensions

widths = {
    "A": 28,  # data_name
    "B": 60,  # path
    "C": 22,  # label
    "D": 32,  # filename
    "E": 10,
    "F": 12,
    "G": 22,
}
for col, width in widths.items():
    ws.column_dimensions[col].width = width

if len(rows) > 0:
    table_range = f"A1:{get_column_letter(len(headers))}{len(rows)+1}"
    table = Table(displayName="ImageManifest", ref=table_range)
    table.tableStyleInfo = TableStyleInfo(
        name="TableStyleMedium2",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=False,
    )
    ws.add_table(table)

# Simple summary sheet
summary = wb.create_sheet("summary")
summary.append(["label", "image_count"])
for label, count in sorted(Counter(r[2] for r in rows).items()):
    summary.append([label, count])
summary.append([])
summary.append(["total_images", len(rows)])
summary.append(["root_folder", str(root)])

for cell in summary[1]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center")
summary.column_dimensions["A"].width = 32
summary.column_dimensions["B"].width = 15

wb.save(output_path)
print(f"Saved: {output_path}")
print(f"Images found: {len(rows)}")
print("Labels:")
for label, count in sorted(Counter(r[2] for r in rows).items()):
    print(f"  {label}: {count}")


Saved: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/organized_image_manifest.xlsx
Images found: 1007
Labels:
  anomaly: 7
  blurry: 3
  occluded: 3
  pathology: 7
  polish_images: 108
  usable_images: 836
  zero_bytes: 43
